# Box-to-box U-Net: xHI sweep

Repeat the demo pipeline at two additional neutral fractions:

- `<x_HI> ≈ 0.82` — near the Tb–δ zero cross-correlation point.
- `<x_HI> ≈ 0.98` — very beginning of the EoR.

Each run gets its own `data_dir` and `ckpt_dir` so the existing
`<x_HI>≈0.5` dataset / checkpoint at
`data/cubes` and `data/checkpoints` is left alone. The z-grid is
widened to bracket xHI=0.98 (which lives at z∼12–14 for these
21cmFAST parameters).

Cost warning: 2 × (20 coevals + 40 U-Net epochs). Hours on CPU.

In [1]:
import sys, pathlib
PHD_ROOT = pathlib.Path().resolve().parents[2]   # .../PhD
sys.path.insert(0, str(PHD_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch

from noisy_reconstruction.code.ml.config   import DemoConfig
from noisy_reconstruction.code.ml.noise    import observe_Tb
from noisy_reconstruction.code.ml.simulate import generate_dataset
from noisy_reconstruction.code.ml.train    import train, load_best

PROJ = PHD_ROOT / 'noisy_reconstruction'

TARGETS = [0.82, 0.98]
# Wider z-grid: xHI=0.98 needs z~12-14, xHI=0.82 needs z~9-10.
Z_GRID = (7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0)

def make_cfg(target_xhi: float) -> DemoConfig:
    tag = f'xhi{int(round(target_xhi*100)):02d}'   # e.g. 'xhi82'
    return DemoConfig(
        target_xHI = target_xhi,
        z_grid     = Z_GRID,
        data_dir   = PROJ / 'data' / 'cubes'       / tag,
        ckpt_dir   = PROJ / 'data' / 'checkpoints' / tag,
    )

cfgs = {x: make_cfg(x) for x in TARGETS}
for x, c in cfgs.items():
    print(f'xHI={x}: data_dir={c.data_dir.relative_to(PROJ)}  ckpt_dir={c.ckpt_dir.relative_to(PROJ)}')

xHI=0.82: data_dir=data/cubes/xhi82  ckpt_dir=data/checkpoints/xhi82
xHI=0.98: data_dir=data/cubes/xhi98  ckpt_dir=data/checkpoints/xhi98


## 1. Generate 21cmFAST cubes for each xHI target

`generate_dataset` scans `z_grid` for the requested `<x_HI>`, then
runs `n_train + n_val` realisations at the chosen z. Cubes already on
disk are skipped, so re-runs are cheap.

In [ ]:
paths = {}
for x, cfg in cfgs.items():
    print(f'\n===== generating dataset for <x_HI>={x} =====')
    paths[x] = generate_dataset(cfg)
    print(f"  -> z used = {paths[x]['z']:.3f},  train={len(paths[x]['train'])}, val={len(paths[x]['val'])}")


===== generating dataset for <x_HI>=0.82 =====
Scanning for z at target x_HI ...


/Users/jelte/Documents/PhD/noisy_reconstruction/.venv/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: Your model (either SOURCE_MODEL=='CONST-ION-EFF' or INTEGRATION_METHOD_X=='GAMMA-APPROX')uses the EPS conditional mass function normalised to the unconditional massfunction provided by the user as matter_options.HMF
  v(inst, attr, value)


  scan: z= 7.00  <x_HI>=0.111
  scan: z= 8.00  <x_HI>=0.464
  scan: z= 9.00  <x_HI>=0.711
  scan: z=10.00  <x_HI>=0.839
  scan: z=11.00  <x_HI>=0.911
  scan: z=12.00  <x_HI>=0.953
  scan: z=13.00  <x_HI>=0.976
  scan: z=14.00  <x_HI>=0.989
  scan: z=15.00  <x_HI>=0.995
  -> z(<x_HI>=0.82) ~ 9.852
[train] seed=1000  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1000.npz  <x_HI>=0.825  Tb_RSD std=6.36 mK  vz std=1093.1 km/s
[train] seed=1001  running 21cmFAST coeval at z=9.852 ...


/Users/jelte/Documents/PhD/noisy_reconstruction/.venv/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: Your model (either SOURCE_MODEL=='CONST-ION-EFF' or INTEGRATION_METHOD_X=='GAMMA-APPROX')uses the EPS conditional mass function normalised to the unconditional massfunction provided by the user as matter_options.HMF
  v(inst, attr, value)


        -> sim_seed1001.npz  <x_HI>=0.824  Tb_RSD std=6.41 mK  vz std=1187.0 km/s
[train] seed=1002  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1002.npz  <x_HI>=0.822  Tb_RSD std=6.31 mK  vz std=1082.5 km/s
[train] seed=1003  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1003.npz  <x_HI>=0.822  Tb_RSD std=6.33 mK  vz std=980.2 km/s
[train] seed=1004  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1004.npz  <x_HI>=0.823  Tb_RSD std=6.35 mK  vz std=1157.6 km/s
[train] seed=1005  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1005.npz  <x_HI>=0.821  Tb_RSD std=6.31 mK  vz std=1088.2 km/s
[train] seed=1006  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1006.npz  <x_HI>=0.823  Tb_RSD std=6.35 mK  vz std=1118.8 km/s
[train] seed=1007  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed1007.npz  <x_HI>=0.824  Tb_RSD std=6.35 mK  vz std=1244.1 km/s
[train] seed=1008  running 21cmFAST coeval at z=9.852 ...
        -> sim_seed10

## 2. Quick look: clean + HERA-observed Tb and true $v_z$ at each xHI

In [ ]:
fig, axes = plt.subplots(len(TARGETS), 4, figsize=(14, 3.4*len(TARGETS)), constrained_layout=True)
for row, x in enumerate(TARGETS):
    cfg = cfgs[x]
    mid = cfg.hii_dim // 2
    sample = np.load(paths[x]['train'][0])
    Tb, xHI, vz = sample['Tb'], sample['xHI'], sample['vz']
    z   = float(sample['z'])
    Tb_obs = observe_Tb(Tb, z, cfg, seed=0)
    print(f"xHI={x}  z={z:.3f}  realized<x_HI>={xHI.mean():.3f}  Tb std={Tb.std():.2f} mK  Tb_obs std={Tb_obs.std():.2f} mK")
    panels = [
        (Tb[:, :, mid],     f'Tb clean  (xHI={x})',    'viridis', None),
        (Tb_obs[:, :, mid], f'Tb observed (xHI={x})',  'viridis', None),
        (xHI[:, :, mid],    f'x_HI',                    'magma',   (0, 1)),
        (vz[:, :, mid],     f'v_z [km/s]',              'RdBu_r',  None),
    ]
    for ax, (data, title, cmap, vrange) in zip(axes[row], panels):
        kw = dict(origin='lower', cmap=cmap)
        if vrange is not None:
            kw.update(vmin=vrange[0], vmax=vrange[1])
        im = ax.imshow(data.T, **kw)
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.85)
plt.show()

## 3. Train a U-Net at each xHI

Same architecture, same HERA noise model; only the training cubes
differ. Each checkpoint lands in its own `ckpt_dir/xhiXX/`.

In [ ]:
history = {}
for x, cfg in cfgs.items():
    print(f'\n===== training for <x_HI>={x} =====')
    history[x] = train(cfg, paths[x]['train'], paths[x]['val'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for x in TARGETS:
    h = history[x]
    axes[0].plot(h['val_loss'], label=f'xHI={x} val')
    axes[0].plot(h['train_loss'], ls=':', alpha=0.6, label=f'xHI={x} train')
    axes[1].plot(h['val_r'], label=f'xHI={x}')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE (standardised)')
axes[0].set_yscale('log'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val Pearson r')
axes[1].axhline(0, color='k', lw=0.5); axes[1].legend()
plt.show()

## 4. Evaluate on a held-out validation cube at each xHI

In [ ]:
def eval_one(cfg, val_path):
    model, stats, device = load_best(cfg)
    val = np.load(val_path)
    Tb_obs  = observe_Tb(val['Tb'], float(val['z']), cfg, seed=999)
    vz_true = val['vz']
    xn = (Tb_obs - stats.x_mean) / stats.x_std
    xn = torch.from_numpy(xn).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(xn).squeeze().cpu().numpy()
    vz_pred = pred * stats.y_std + stats.y_mean
    r = float(np.corrcoef(vz_true.ravel(), vz_pred.ravel())[0, 1])
    return dict(Tb_obs=Tb_obs, vz_true=vz_true, vz_pred=vz_pred, r=r, z=float(val['z']))

evals = {x: eval_one(cfgs[x], paths[x]['val'][0]) for x in TARGETS}
for x, e in evals.items():
    print(f'xHI={x}  z={e["z"]:.3f}  voxel Pearson r = {e["r"]:+.3f}')

In [ ]:
fig, axes = plt.subplots(len(TARGETS), 3, figsize=(11, 3.4*len(TARGETS)), constrained_layout=True)
for row, x in enumerate(TARGETS):
    e = evals[x]
    mid = cfgs[x].hii_dim // 2
    vmax = float(np.percentile(np.abs(e['vz_true']), 99))
    panels = [
        (e['vz_true'][:, :, mid],                            f'v_z true  (xHI={x})'),
        (e['vz_pred'][:, :, mid],                            f"v_z pred  r={e['r']:+.3f}"),
        (e['vz_pred'][:, :, mid] - e['vz_true'][:, :, mid],  'pred - true'),
    ]
    for ax, (data, title) in zip(axes[row], panels):
        im = ax.imshow(data.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.85)
plt.show()

### 1D power-spectrum comparison (true vs U-Net prediction)

In [ ]:
def _power_1d(cube, box_len):
    n = cube.shape[0]
    k1 = np.fft.fftfreq(n, d=box_len / n) * 2 * np.pi
    kx, ky, kz = np.meshgrid(k1, k1, k1, indexing='ij')
    kmag = np.sqrt(kx**2 + ky**2 + kz**2)
    fft  = np.fft.fftn(cube - cube.mean())
    pk3  = (np.abs(fft)**2) * (box_len / n)**3 / (n**3)
    bins = np.linspace(kmag[kmag > 0].min(), kmag.max(), 25)
    k_c  = 0.5 * (bins[1:] + bins[:-1])
    idx  = np.digitize(kmag.ravel(), bins)
    pk1  = np.array([pk3.ravel()[idx == i].mean() if np.any(idx == i) else np.nan
                     for i in range(1, len(bins))])
    return k_c, pk1

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5*len(TARGETS), 3.5), constrained_layout=True)
for ax, x in zip(np.atleast_1d(axes), TARGETS):
    e = evals[x]
    k, pk_true = _power_1d(e['vz_true'], cfgs[x].box_len)
    _, pk_pred = _power_1d(e['vz_pred'], cfgs[x].box_len)
    ax.loglog(k, pk_true, label='true')
    ax.loglog(k, pk_pred, ls='--', label='pred')
    ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')
    ax.set_ylabel(r'$P_{v_z}(k)\ [(\mathrm{km/s})^2\,\mathrm{Mpc}^3]$')
    ax.set_title(f'xHI={x}  (r={e["r"]:+.3f})')
    ax.legend()
plt.show()